# Notebook 2: Fine-tuning NLLB-200-distilled-600M (transfer learning)
WMT26 Indic MT, low-resource NMT project. Runs on **Kaggle 2×T4** (two directions in parallel, fp16, about 12 GB/GPU) or **Colab A100** (bf16).

Estimated time: about 4 h on 2×T4, about 1.5 h on A100. Independent of Notebook 1, so both can run at the same time on different accounts.

**How to run**

1. On your laptop: `python scripts/make_bundle.py` creates `dist/nmt_bundle.zip`.
2. **Kaggle:** New Notebook, then File, then Import Notebook (this file). Next, Add Input, then Upload Dataset (`nmt_bundle.zip`).
   Settings: **Accelerator = GPU T4 x2**, **Internet = On**. Click **Save Version, then Save & Run All (Commit)** so it
   keeps running after you close the tab. When it finishes, download `nb2_results.zip` (and `_checkpoints.zip`) from Output.
3. **Colab:** Upload this notebook, put `nmt_bundle.zip` in *My Drive* (root), Runtime, then **A100 GPU**, then Run all.
   Outputs are written straight to `MyDrive/nmt_outputs/`, so if Colab disconnects, just run all again and finished runs are skipped.
4. On your laptop, extract the results zip in the project root, then `bash run.sh eval`.

In [ ]:
# ---- 1. Locate the project bundle and unpack it (works on Kaggle and Colab) ----
import os, sys, glob, zipfile, shutil, subprocess
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# NB: recent Colab images also contain a /kaggle folder, so check Colab first
IN_KAGGLE = (not IN_COLAB) and ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input"))
WORK = "/kaggle/working/project" if IN_KAGGLE else "/content/project"

def find_bundle():
    pats = ["/kaggle/input/**/nmt_bundle.zip", "/content/nmt_bundle.zip",
            "/content/drive/MyDrive/nmt_bundle.zip", "/content/drive/MyDrive/**/nmt_bundle.zip"]
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return ("zip", hits[0])
    # Kaggle auto-extracts uploaded zips -> look for an extracted project folder
    for d in glob.glob("/kaggle/input/**/src/models", recursive=True):
        return ("dir", os.path.dirname(os.path.dirname(d)))
    return (None, None)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

kind, path = find_bundle()
if kind is None and IN_COLAB:
    from google.colab import files
    print("Upload dist/nmt_bundle.zip"); up = files.upload(); kind, path = "zip", "/content/" + list(up)[0]
assert kind, "nmt_bundle.zip not found - attach it as a Kaggle dataset / put it in MyDrive"
if not os.path.exists(WORK + "/src"):
    os.makedirs(WORK, exist_ok=True)
    if kind == "zip":
        zipfile.ZipFile(path).extractall(WORK)
    else:
        shutil.copytree(path, WORK, dirs_exist_ok=True)
os.chdir(WORK); sys.path.insert(0, WORK)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # less fragmentation (inherited by jobs)

# On Colab keep outputs on Google Drive so a disconnect never loses finished runs
if IN_COLAB:
    drive_out = "/content/drive/MyDrive/nmt_outputs"
    os.makedirs(drive_out, exist_ok=True)
    if not os.path.islink("outputs"):
        if os.path.isdir("outputs"):
            shutil.copytree("outputs", drive_out, dirs_exist_ok=True); shutil.rmtree("outputs")
        os.symlink(drive_out, "outputs")
print("bundle:", path, "| workdir:", os.getcwd())

In [ ]:
# ---- 2. Dependencies + GPU check ----
!pip install -q -r requirements-gpu.txt
import torch
n = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
print("bf16 supported:", torch.cuda.is_bf16_supported() if n else None)
assert n > 0, "No GPU! Kaggle: Settings -> Accelerator -> GPU T4 x2.  Colab: Runtime -> Change runtime type -> A100"

In [ ]:
# ---- helpers: skip finished runs (safe to re-run the notebook after a disconnect) ----
import json
from pathlib import Path
from src.run_queue import run_jobs

def finished(model, direction):
    f = Path("outputs") / model / direction / "run_config.json"
    return f.exists() and json.loads(f.read_text()).get("finished", False)

def todo(jobs):
    return [(n, a) for (n, a, m, d) in jobs if not finished(m, d)]

DIRECTIONS = ["en-as", "as-en", "en-lus", "lus-en", "en-mni", "mni-en", "en-bodo", "bodo-en"]  # big pairs first

In [ ]:
# ---- 3. 2-minute smoke test (catches library / memory problems before the long run) ----
r = subprocess.run([sys.executable, "-m", "src.models.advanced.nllb_finetune", "--direction", "en-bodo",
                    "--model-name", "_nllb_smoke", "--max-steps", "10", "--eval-every", "5", "--limit-train", "400"],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, "smoke test failed - see the error above"
shutil.rmtree("outputs/_nllb_smoke", ignore_errors=True)

In [ ]:
# ---- 4. Fine-tune NLLB-200-distilled-600M, 8 directions ----
jobs = [(f"nllb_{d}", ["-m", "src.models.advanced.nllb_finetune", "--direction", d], "nllb_finetune", d)
        for d in DIRECTIONS]
print(run_jobs(todo(jobs)))
for d in DIRECTIONS:
    f = Path("outputs/nllb_finetune") / d / "run_config.json"
    if f.exists():
        r = json.loads(f.read_text())
        print(f"{d:8s} zero-shot dev chrF++ {r.get('zeroshot_dev_chrf', float('nan')):6.2f} -> fine-tuned "
              f"{r.get('dev_chrf_beam', float('nan')):6.2f}   {r['wall_clock_s']/60:.0f} min")

In [ ]:
# ---- COMET (optional; installed last so it cannot break the training environment) ----
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet"], capture_output=True, text=True)
print("pip:", "OK" if r.returncode == 0 else r.stderr[-3000:])
env = dict(os.environ, USE_TF="0", TRANSFORMERS_NO_TF="1")
r = subprocess.run([sys.executable, "-m", "src.eval.comet_score", "--models"] + ["nllb_finetune"], capture_output=True, text=True, env=env)
print(r.stdout[-3000:])
if r.returncode != 0:
    print("COMET failed (non-fatal; chrF++/BLEU are computed locally). Error tail:")
    print(r.stderr[-4000:])

In [ ]:
# ---- Package results: small results zip + separate checkpoints zip ----
import zipfile, os
from pathlib import Path
out_dir = "/kaggle/working" if IN_KAGGLE else "/content/drive/MyDrive"
res = f"{out_dir}/nb2_results.zip"; ck = f"{out_dir}/nb2_checkpoints.zip"
with zipfile.ZipFile(res, "w", zipfile.ZIP_DEFLATED) as zr, zipfile.ZipFile(ck, "w") as zc:
    for root in ["nllb_finetune"]:
        for f in Path("outputs", root).rglob("*") if Path("outputs", root).exists() else []:
            if f.is_file():
                (zc if f.suffix == ".pt" or "checkpoint" in f.parts else zr).write(f, f.as_posix())
    for f in Path("outputs/logs").glob("*.log"):
        zr.write(f, f.as_posix())
for z in (res, ck):
    print(z, round(os.path.getsize(z) / 1e6, 1), "MB")
print("Download the zip(s) and extract them in the project root (they contain outputs/...).")